# 07 · Trees

> Quasi ogni problema sugli alberi binari è una **DFS ricorsiva** (cosa restituisco al padre? cosa passo ai figli?) o una **BFS per livelli** con una coda.
> Idee per i singoli problemi: [Big-O › Alberi e BST](../Fundamentals/Big-O%20e%20strutture%20dati.md#alberi-e-bst)

## Problemi → cosa ti serve

| Problema | Concetti | Sintassi |
|---|---|---|
| Invert Binary Tree | modifica ricorsiva | scambio `node.left, node.right = node.right, node.left` |
| Maximum Depth of Binary Tree | DFS che **restituisce** un valore | `1 + max(dfs(l), dfs(r))` |
| Diameter of Binary Tree | restituisci l'altezza, aggiorna la risposta globale | `nonlocal best` o `self.best` |
| Balanced Binary Tree | altezza + flag (o `-1` come "sbilanciato") | `abs(hl - hr) > 1` |
| Same Tree | confronto ricorsivo di due alberi | `if not p and not q`, `p.val == q.val and ...` |
| Subtree of Another Tree | Same Tree da ogni nodo | ricorsione dentro ricorsione |
| Lowest Common Ancestor of a BST | proprietà del BST: scendi a sinistra o a destra | ciclo `while` |
| Binary Tree Level Order Traversal | BFS, un livello per giro | `deque`, `for _ in range(len(queue))` |
| Binary Tree Right Side View | ultimo nodo di ogni livello | `level[-1]` |
| Count Good Nodes in Binary Tree | DFS che **riceve** il massimo del percorso | parametro extra `max_so_far` |
| Validate Binary Search Tree | intervallo valido passato ai figli | `low < node.val < high`, `float("inf")` |
| Kth Smallest Element in a BST | inorder = valori in ordine | inorder iterativo con stack |
| Construct Binary Tree from Preorder and Inorder | radice = primo del preorder; inorder la divide | `{v: i for i, v in enumerate(inorder)}` |
| Binary Tree Maximum Path Sum | come il diametro, con somme e `max(0, ...)` | `nonlocal`, `float("-inf")` |
| Serialize and Deserialize Binary Tree | preorder con marcatore per `None` | `",".join`, `split`, `iter`, `next` |

## Il nodo e un costruttore di prova

LeetCode rappresenta gli alberi come lista per livelli (`[3, 9, 20, None, None, 15, 7]`). Questo helper la converte in nodi per fare prove locali.

In [1]:
from collections import deque
from typing import Optional

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def build_tree(values: list) -> Optional[TreeNode]:
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    items = iter(values[1:])
    while queue:
        node = queue.popleft()
        for side in ("left", "right"):
            val = next(items, None)
            if val is not None:
                child = TreeNode(val)
                setattr(node, side, child)
                queue.append(child)
    return root

root = build_tree([3, 9, 20, None, None, 15, 7])
print(root.val, root.left.val, root.right.left.val)  # -> 3 9 15

3 9 15


## DFS: cosa restituisci al padre

Pensa **solo al nodo corrente**: assumi che la chiamata sui figli restituisca la risposta corretta per il loro sottoalbero, e combina. Caso base: `None`.

In [2]:
# continua dall'esempio precedente
def height(node: Optional[TreeNode]) -> int:
    if not node:
        return 0
    return 1 + max(height(node.left), height(node.right))

def total(node: Optional[TreeNode]) -> int:
    if not node:
        return 0
    return node.val + total(node.left) + total(node.right)

print(height(root), total(root))  # -> 3 54

3 54


## DFS: risposta globale con `nonlocal`

Spesso la funzione ricorsiva restituisce una cosa (l'altezza, il ramo migliore) ma la risposta è un'altra (diametro, somma del percorso), da aggiornare in ogni nodo. Usa una variabile esterna e `nonlocal` per poterla riassegnare. In una classe LeetCode puoi usare anche `self.best`.

In [3]:
# continua dall'esempio precedente
def longest_path_edges(root: Optional[TreeNode]) -> int:
    best = 0

    def depth(node: Optional[TreeNode]) -> int:
        nonlocal best  # senza: UnboundLocalError alla riga best = ...
        if not node:
            return 0
        left, right = depth(node.left), depth(node.right)
        best = max(best, left + right)  # percorso che passa per questo nodo
        return 1 + max(left, right)     # al padre serve un solo ramo

    depth(root)
    return best

print(longest_path_edges(root))  # -> 3 (9 → 3 → 20 → 15)

3


⚠️ `nonlocal` serve solo per **riassegnare**. Se la variabile esterna è una lista e fai `result.append(...)`, non serve.

## DFS: cosa passi ai figli

Quando il nodo ha bisogno di informazioni dagli antenati (massimo sul percorso, limiti validi), passale come parametri. Per i BST: ogni nodo deve stare in un intervallo aperto `(low, high)` che si restringe scendendo.

In [4]:
# continua dall'esempio precedente
def within_bounds(node: Optional[TreeNode], low=float("-inf"), high=float("inf")) -> bool:
    if not node:
        return True
    if not (low < node.val < high):
        return False
    return within_bounds(node.left, low, node.val) and within_bounds(node.right, node.val, high)

bst = build_tree([5, 3, 8, 1, 4])
print(within_bounds(bst), within_bounds(root))  # -> True False

True False


## Due alberi insieme

Il caso base copre tre situazioni: entrambi `None` (uguali), uno solo `None` (diversi), entrambi presenti (confronta e ricorri).

In [5]:
# continua dall'esempio precedente
def same(p: Optional[TreeNode], q: Optional[TreeNode]) -> bool:
    if not p and not q:
        return True
    if not p or not q:
        return False
    return p.val == q.val and same(p.left, q.left) and same(p.right, q.right)

print(same(build_tree([1, 2]), build_tree([1, 2])), same(build_tree([1, 2]), build_tree([1, None, 2])))  # -> True False

True False


## BFS per livelli

All'inizio di ogni giro la coda contiene **esattamente** il livello corrente: `len(queue)` ti dice quanti nodi estrarre.

In [6]:
# continua dall'esempio precedente
def levels(root: Optional[TreeNode]) -> list[list[int]]:
    if not root:
        return []
    result = []
    queue = deque([root])
    while queue:
        level = []
        for _ in range(len(queue)):  # len valutato una volta sola, all'inizio
            node = queue.popleft()
            level.append(node.val)
            if node.left:
                queue.append(node.left)
            if node.right:
                queue.append(node.right)
        result.append(level)
    return result

print(levels(root))                           # -> [[3], [9, 20], [15, 7]]
print([level[-1] for level in levels(root)])  # -> [3, 20, 7]: vista da destra

[[3], [9, 20], [15, 7]]
[3, 20, 7]


## Inorder iterativo (BST in ordine)

In un BST la visita inorder (sinistra, nodo, destra) produce i valori ordinati. La versione iterativa con uno stack esplicito ti permette di fermarti al k-esimo senza visitare tutto.

In [7]:
# continua dall'esempio precedente
def inorder(root: Optional[TreeNode]) -> list[int]:
    out, stack, node = [], [], root
    while stack or node:
        while node:            # scendi tutto a sinistra
            stack.append(node)
            node = node.left
        node = stack.pop()     # il più piccolo non ancora visitato
        out.append(node.val)   # qui potresti contare e fermarti al k-esimo
        node = node.right
    return out

print(inorder(bst))  # -> [1, 3, 4, 5, 8]

[1, 3, 4, 5, 8]


## Costruire alberi: indici e dict di posizioni

Per dividere l'inorder intorno alla radice senza cercarla ogni volta (`list.index` è `O(n)`), precalcola valore → indice. Lavora con indici `(start, end)` invece di slicing per evitare copie.

In [8]:
inorder_vals = [9, 3, 15, 20, 7]
position = {val: i for i, val in enumerate(inorder_vals)}
root_val = 3  # il primo del preorder
mid = position[root_val]
print(mid, inorder_vals[:mid], inorder_vals[mid + 1 :])  # -> 1 [9] [15, 20, 7]: sinistra e destra

1 [9] [15, 20, 7]


## Serializzare: `join`, `split`, `iter`/`next`

Un iteratore consuma i token uno alla volta: ogni chiamata ricorsiva prende il "prossimo" senza dover passare un indice.

In [9]:
tokens = "1,2,#,#,3,#,#".split(",")
it = iter(tokens)
print(next(it), next(it), next(it))  # -> 1 2 #
print(next(it, None), list(it))      # -> # ['3', '#', '#']: next con default non solleva StopIteration
print(",".join(["1", "#"]))          # -> 1,#

1 2 #
# ['3', '#', '#']
1,#


## Errori comuni

- Dimenticare il caso base `if not node`.
- Ricorsione profonda (alberi degeneri di 10⁴+ nodi): `sys.setrecursionlimit(10**5)` o versione iterativa.
- Validate BST controllando solo i figli diretti: serve l'intervallo ereditato da tutti gli antenati.
- Valori negativi nella Max Path Sum: inizializza con `float("-inf")` e scarta i rami negativi con `max(0, ramo)`.
- `if node.val:` è falso per `val == 0`: confronta con `is None` quando conta.